In [ ]:
from PIL import Image
from pix2tex.cli import LatexOCR

# 1. Initialize the model
print("Loading Pix2Tex model...")
model = LatexOCR()
print("Model loaded successfully!")

# 2. Load our test image
image_path = "../data/raw_images/test.png"

try:
    img = Image.open(image_path)
    print("Image loaded. Running OCR...")
    
    # 3. Perform OCR
    latex_output = model(img)
    
    print("-" * 30)
    print("Predicted LaTeX Output:")
    print(latex_output)
    print("-" * 30)
    
except FileNotFoundError:
    print(f"Error: Could not find image at {image_path}. Did you name it correctly?")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
from PIL import Image
from pix2tex.cli import LatexOCR

# 1. Initialize the model
model = LatexOCR()

# 2. Define test set
test_cases = [
    {"path": "../data/raw_images/test1.png", "expected": r"\frac{a}{b}"},
    {"path": "../data/raw_images/test2.png", "expected": r"\int_0^1 x^2 dx"},
    {"path": "../data/raw_images/test3.png", "expected": r"\sqrt{x^2 + y^2}"},
]

# 3. Run predictions and measure accuracy
correct = 0
total = len(test_cases)

for case in test_cases:
    try:
        img = Image.open(case["path"])
        prediction = model(img).strip()
        expected = case["expected"].strip()

        print(f"Image: {case['path']}")
        print(f"Predicted: {prediction}")
        print(f"Expected:  {expected}")

        if prediction == expected:
            print("✅ Correct\n")
            correct += 1
        else:
            print("❌ Incorrect\n")

    except Exception as e:
        print(f"Error with {case['path']}: {e}")

# 4. Print accuracy
accuracy = correct / total * 100
print(f"Model Accuracy: {accuracy:.2f}% ({correct}/{total})")


In [ ]:
import sys
sys.path.append('../src') # Allows us to import from the src folder
from ocr_engine.preprocessing import preprocess_image
import matplotlib.pyplot as plt

raw_image_path = "../data/raw_images/test.png"
processed_image_path = "../data/processed_images/test_cleaned.png"

# Run our new function
cleaned_img = preprocess_image(raw_image_path, processed_image_path)

# Display the result
plt.imshow(cleaned_img, cmap='gray')
plt.title("Cleaned Image")
plt.axis('off')
plt.show()

In [ ]:
import sys
sys.path.append('../src')
from ocr_engine.preprocessing import preprocess_image
from ocr_engine.ocr_model import MathOCR
import matplotlib.pyplot as plt

# 1. Preprocess the raw image
raw_path = "../data/raw_images/test.png"
cleaned_image_array = preprocess_image(raw_path)

# 2. VISUAL DEBUGGING: Let's see what OpenCV actually did
print(f"Cropped Image Size: {cleaned_image_array.shape}")
plt.figure(figsize=(6, 3))
plt.imshow(cleaned_image_array, cmap='gray')
plt.title("What the AI actually sees")
plt.axis('off')
plt.show()

# 3. Predict (We will wrap this in a try-except just in case)
ocr_system = MathOCR()
print("Running prediction on the cropped image...")
final_latex = ocr_system.predict(cleaned_image_array)

print("\n--- Final Pipeline Output ---")
print(final_latex)

In [ ]:
import sys
sys.path.append('../src')
from ocr_engine.latex_parser import EquationParser
import json

# Replace this with the successful LaTeX output you got from your textbook image
sample_latex = "{\frac{d f}{d x}}=\operatorname*{lim}_{h\to0}{\frac{f(x+h)-f(x)}{h}}" 

# Initialize and parse
parser = EquationParser()
parsed_data = parser.parse_to_dict(sample_latex)

# Print it beautifully as JSON
print(json.dumps(parsed_data, indent=4))

In [ ]:
import sys 
import json 
sys.path.append('../src')
from ocr_engine.latex_parser import EquationParser\

sample_latex=r"{\frac{d f}{d x}}=\operatorname*{lim}_{h\to0}{\frac{f(x+h)-f(x)}{h}}"

parser = EquationParser()
parsed_data = parser.parse_to_dict(sample_latex)
print(json.dumps(parsed_data,indent=4))

In [ ]:
import sys 
sys.path.append('../src/backend')
from database import save_equation, client

# 1. The data we just successfully parsed
parsed_data = {
    "latex": r"{\frac{d f}{d x}}=\operatorname*{lim}_{h\to0}{\frac{f(x+h)-f(x)}{h}}",
    "sympy_format": "Eq(Derivative(f, x), operatorname*(l*(i*m)))",
    "type": "equation"
}

# 2. Check if MongoDB is actually running before we try to save
try:
    client.admin.command('ping')
    print("Successfully connected to MongoDB!")

    #3. Save the data
    inserted_id = save_equation(parsed_data)
    print(f"Equation saved successsfully with Database ID:{inserted_id}")

except Exception as e:
    print("\nWARNING : Could not connect to MOngoDB")
    print("Is MonogDB Community Server running on your machine?")